In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


In [2]:
# --- your paths (same as notebook) ---
SRC_STREMLINE = Path("/Users/921623492/Abdoul Thesis/src/STREAMLINE")
ORIGINAL_DATA = SRC_STREMLINE / "data"
OUT = Path("/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output")
TRAIN_NAME = "SNP_merged_train"

CV_DIR = OUT / TRAIN_NAME / "CVDatasets"
MODEL_DIR = OUT / TRAIN_NAME / "models" / "pickledModels"


In [3]:
snp_train = ORIGINAL_DATA / "SNPData"
snp_test = ORIGINAL_DATA / "SNPRepData"
snp_train_data = snp_train / "SNP_merged_train.csv"
snp_test_data = snp_test / "SNP_merged_test.csv" # untouched 

snp_train_data = pd.read_csv(snp_train_data)
snp_test_data = pd.read_csv(snp_test_data)


In [4]:
# encode label 
snp_test_data['label'] = snp_test_data['label'].astype(str).str.strip().map({"R": 0, "S": 1})


In [5]:
snp_test_processed = pd.read_csv( OUT / 'SNP_merged_train/replication/SNP_merged_test/SNP_merged_test_Processed.csv')
snp_test_processed.head()


,sample_id,SNP_0,SNP_1,SNP_2,SNP_3,SNP_4,SNP_5,SNP_6,SNP_7,SNP_8,...,SNP_56699_0,SNP_56699_1,SNP_56699_2,SNP_56700_0,SNP_56700_1,SNP_56700_2,SNP_56700_3,SNP_56716_0,SNP_56716_1,SNP_56716_2
0,ERR1218582,0,0,0,1,1,0,1,1,1,...,False,True,False,False,True,False,0,0,0,0
1,ERR1218605,0,0,0,1,0,0,1,0,0,...,False,False,True,False,False,True,0,0,0,0
2,ERR1218607,0,0,0,1,1,0,1,1,1,...,False,True,False,False,True,False,0,0,0,0
3,ERR1218623,0,0,0,1,1,0,1,1,1,...,False,True,False,False,True,False,0,0,0,0
4,ERR1218646,0,1,1,1,0,0,1,0,0,...,False,True,False,False,True,False,0,0,0,0


In [6]:
## merging label to processed test data 
snp_test_processed = pd.merge(snp_test_processed, snp_test_data[['sample_id', 'label']], on='sample_id', how='left')
snp_test_processed.head()


,sample_id,SNP_0,SNP_1,SNP_2,SNP_3,SNP_4,SNP_5,SNP_6,SNP_7,SNP_8,...,SNP_56699_1,SNP_56699_2,SNP_56700_0,SNP_56700_1,SNP_56700_2,SNP_56700_3,SNP_56716_0,SNP_56716_1,SNP_56716_2,label_y
0,ERR1218582,0,0,0,1,1,0,1,1,1,...,True,False,False,True,False,0,0,0,0,0
1,ERR1218605,0,0,0,1,0,0,1,0,0,...,False,True,False,False,True,0,0,0,0,0
2,ERR1218607,0,0,0,1,1,0,1,1,1,...,True,False,False,True,False,0,0,0,0,0
3,ERR1218623,0,0,0,1,1,0,1,1,1,...,True,False,False,True,False,0,0,0,0,0
4,ERR1218646,0,1,1,1,0,0,1,0,0,...,True,False,False,True,False,0,0,0,0,0


In [7]:
cv0_train_path = CV_DIR / f"{TRAIN_NAME}_CV_0_Train.csv"  # uses your existing CV_DIR/TRAIN_NAME


In [8]:
# Load CV train header
cv_cols = list(pd.read_csv(cv0_train_path, nrows=0).columns)
train_feature_cols = [c for c in cv_cols if c not in ["label", "sample_id"]]


In [9]:
len(cv_cols), len(train_feature_cols)


(5002, 5000)

In [10]:
# Extra features after took the top 5000 features 
extra_feats = [
    c for c in snp_test_processed.columns
    if c not in train_feature_cols + ["sample_id", "label"]
]
print("Extra features in processed test:", len(extra_feats))


Extra features in processed test: 62261


In [11]:
X = snp_test_processed[train_feature_cols].to_numpy()
y = snp_test_processed['label_y'].to_numpy()

X.shape, y.shape


((118, 5000), (118,))

In [12]:
# cv_paths = sorted(Path(CV_DIR).glob(f"{TRAIN_NAME}_CV_*_Train.csv"))
# for cv_path in cv_paths:
#     cols = pd.read_csv(cv_path, nrows=0).columns
#     feats = [c for c in cols if c not in ["label", "sample_id"]]
#     print(f"{cv_path.name}: {len(feats)} features, hash={hash(tuple(feats))}")


In [13]:
def build_test_matrix_for_fold(test_encoded, cv_train_csv, label_col="label_y"):
    """
    Align processed test data to the feature space of a specific CV fold.

    test_encoded: processed test dataframe (has sample_id, label_raw, all 67k features)
    cv_train_csv: path to e.g. SNP_merged_train_CV_0_Train.csv
    label_col:    column in test_encoded with numeric labels (0/1)
    """
    # Get fold-specific feature list
    cv_cols = pd.read_csv(cv_train_csv, nrows=0).columns
    feat_cols = [c for c in cv_cols if c not in ["label", "sample_id"]]

    # Check all features are present in test
    missing = [c for c in feat_cols if c not in test_encoded.columns]
    if missing:
        raise ValueError(
            f"{cv_train_csv} expects {len(feat_cols)} features, "
            f"but {len(missing)} are missing in test_encoded. "
            f"First few missing: {missing[:20]}"
        )

    # Build X, y, ids
    X = test_encoded[feat_cols].to_numpy()
    y = test_encoded[label_col].astype(int).to_numpy()
    ids = test_encoded["sample_id"].astype(str).to_numpy()

    return X, y, ids, feat_cols


In [14]:
## X folds and y folds
cv_paths = sorted(Path(CV_DIR).glob(f"{TRAIN_NAME}_CV_*_Train.csv"))

for cv_path in cv_paths:
    print(f"\n=== {cv_path.name} ===")
    X_fold, y_fold, ids_fold, feat_cols_fold = build_test_matrix_for_fold(snp_test_processed, cv_path)
    print("X shape:", X_fold.shape, "y shape:", y_fold.shape)



=== SNP_merged_train_CV_0_Train.csv ===
X shape: (118, 5000) y shape: (118,)

=== SNP_merged_train_CV_1_Train.csv ===
X shape: (118, 5000) y shape: (118,)

=== SNP_merged_train_CV_2_Train.csv ===
X shape: (118, 5000) y shape: (118,)


In [15]:
list(MODEL_DIR.iterdir())


[PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/RF_2.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/LR_2.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/XGB_0.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/RF_1.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/RF_0.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/KNN_2.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/KNN_1.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged

In [16]:
algo = "LR"
CV = 0 

model_path = MODEL_DIR / f"{algo}_{CV}.pickle"
model = joblib.load(model_path)


In [ ]:
model


LogisticRegression(C=2090.9008134216406, max_iter=730, random_state=42,
                   solver='newton-cg')

In [18]:
model.intercept_


array([-0.00558306])

In [19]:
# coefs of the model 
model.coef_


array([[-3.70529197,  5.62100787, -5.56258258, ..., -0.21803361,
         0.72550362,  0.02721665]])

In [ ]:
### Model Optimization 

